# Model Evaluation

## Objective

The objective of this notebook is to evaluate the performance of the Isolation Forest model for detecting anomalous household energy consumption.

Since Isolation Forest is an unsupervised learning algorithm, evaluation focuses on anomaly distribution, anomaly scores, visualization, and qualitative assessment rather than traditional supervised classification metrics.

## Inject Synthetic Anomalies

In [ ]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import joblib

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay
)

import warnings
warnings.filterwarnings("ignore")

## Evaluation Dataset

The evaluation dataset contains:

- Engineered features
- Standardized numerical variables
- Predicted anomaly labels
- Anomaly scores

These outputs are used to assess the effectiveness of the anomaly detection pipeline.

In [ ]:
raw_df = pd.read_csv("data/processed/cleaned_energy_data.csv",parse_dates=["datetime"])

## Model Overview

The trained Isolation Forest model developed in the previous notebook is loaded for evaluation.

This notebook analyzes the behaviour of the trained model, examines the generated anomaly labels, and assesses the quality of the detected abnormal observations.

In [ ]:
import joblib 
sc= joblib.load("models/standard_scaler.pkl")
iso_model= joblib.load("models/isolation_forest.pkl")

In [ ]:
feature_columns = [

    # Original electrical variables
    "Global_active_power",
    "Global_reactive_power",
    "Voltage",
    "Global_intensity",
    "Sub_metering_1",
    "Sub_metering_2",
    "Sub_metering_3",

    # Temporal
    "hour_sin",
    "hour_cos",
    "day_sin",
    "day_cos",
    "year_sin",
    "year_cos",

    # Lag
    "active_power_lag_1",
    "active_power_lag_5",
    "active_power_lag_15",
    "active_power_lag_60",

    # Rolling
    "active_power_rolling_mean_15",
    "active_power_rolling_std_15",
    "active_power_rolling_min_15",
    "active_power_rolling_max_15",

    "active_power_rolling_mean_60",
    "active_power_rolling_std_60",
    "active_power_rolling_min_60",
    "active_power_rolling_max_60",

    # Behaviour
    "active_power_change_1",
    "deviation_from_15min_mean",
    "deviation_from_60min_mean",

    # Log transformed
    "active_power_change_rate_log",
    "rolling_zscore_15_log",
    "rolling_zscore_60_log"
]

In [ ]:
print("Dataset Shape :", raw_df.shape)

print()

print(type(iso_model))

print(type(sc))

print()

print("Number of Features :", len(feature_columns))

## Prediction Results

Each observation has been assigned one of the following labels:

- **0** → Normal Observation
- **1** → Anomalous Observation

The anomaly labels provide an initial indication of unusual household energy consumption.

In [ ]:
evaluation_df = raw_df.copy()

evaluation_df["true_anomaly"] = 0
evaluation_df.head()
evaluation_df["true_anomaly"].value_counts()

## Distribution of Predictions

Examining the proportion of normal and anomalous observations provides an overview of model behaviour.

A relatively small percentage of anomalies is generally expected because abnormal events occur infrequently in real-world household electricity consumption.

In [ ]:
np.random.seed(42)

power_indices = np.random.choice(
    evaluation_df.index,
    size=int(0.003 * len(evaluation_df)),
    replace=False
)

evaluation_df.loc[
    power_indices,
    "true_anomaly"
] = 1

evaluation_df.loc[
    power_indices,
    "Global_active_power"
] *= np.random.uniform(
    2.5,
    4.0,
    len(power_indices)
)

evaluation_df.loc[
    power_indices,
    "Global_intensity"
] *= np.random.uniform(
    2.0,
    3.0,
    len(power_indices)
)

In [ ]:
evaluation_df["true_anomaly"].value_counts()

## Anomaly Score Analysis

In addition to binary predictions, Isolation Forest computes a continuous anomaly score.

Lower scores indicate observations that are more isolated within the feature space and therefore more likely to represent abnormal energy consumption.

In [ ]:
remaining = evaluation_df.index.difference(power_indices)

voltage_indices = np.random.choice(
    remaining,
    size=int(0.002 * len(evaluation_df)),
    replace=False
)

evaluation_df.loc[
    voltage_indices,
    "true_anomaly"
] = 1

evaluation_df.loc[
    voltage_indices,
    "Voltage"
] -= np.random.uniform(
    20,
    35,
    len(voltage_indices)
)

In [ ]:
evaluation_df["true_anomaly"].value_counts()

In [ ]:
remaining = remaining.difference(voltage_indices)

current_indices = np.random.choice(
    remaining,
    size=int(0.002 * len(evaluation_df)),
    replace=False
)

evaluation_df.loc[
    current_indices,
    "true_anomaly"
] = 1

evaluation_df.loc[
    current_indices,
    "Global_intensity"
] *= np.random.uniform(
    3,
    5,
    len(current_indices)
)

In [ ]:
evaluation_df["true_anomaly"].value_counts()

## Temporal Analysis of Anomalies

Visualizing anomaly predictions over time helps determine whether abnormal events occur randomly or follow recurring temporal patterns.

This analysis provides additional confidence in the practical usefulness of the model.

In [ ]:
evaluation_df["hour"] = evaluation_df["datetime"].dt.hour
evaluation_df["day_of_week"] = evaluation_df["datetime"].dt.dayofweek
evaluation_df["month"] = evaluation_df["datetime"].dt.month
evaluation_df["day_of_year"] = evaluation_df["datetime"].dt.dayofyear
evaluation_df["is_weekend"] = (evaluation_df["day_of_week"] >= 5).astype(int)

In [ ]:
evaluation_df["hour_sin"] = np.sin(2*np.pi*evaluation_df["hour"]/24)
evaluation_df["hour_cos"] = np.cos(2*np.pi*evaluation_df["hour"]/24)
evaluation_df["day_sin"] = np.sin(2*np.pi*evaluation_df["day_of_week"]/7)
evaluation_df["day_cos"] = np.cos(2*np.pi*evaluation_df["day_of_week"]/7)
evaluation_df["year_sin"] = np.sin(2*np.pi*evaluation_df["day_of_year"]/365.25)
evaluation_df["year_cos"] = np.cos(2*np.pi*evaluation_df["day_of_year"]/365.25)


In [ ]:
primary_signal = "Global_active_power"
print("Primary energy signal:",primary_signal)
evaluation_df[primary_signal].describe()

In [ ]:
lag_periods = [1, 5, 15, 60]

for lag in lag_periods:
    evaluation_df[f"active_power_lag_{lag}"] = (
        evaluation_df.groupby("segment_id")[primary_signal]
        .shift(lag)
    )

In [ ]:
lag_columns = [f"active_power_lag_{lag}" for lag in lag_periods]
evaluation_df[lag_columns].isnull().sum()

In [ ]:
historical_power = (evaluation_df.groupby("segment_id")[primary_signal].shift(1))

In [ ]:
rolling_windows =[15,60]
for window in rolling_windows :
    evaluation_df[f"active_power_rolling_mean_{window}"] = (historical_power.groupby(evaluation_df["segment_id"]).rolling(window=window,min_periods=window).mean().reset_index(level=0,drop=True))
    evaluation_df[f"active_power_rolling_std_{window}"] = (historical_power.groupby(evaluation_df["segment_id"]).rolling(window=window,min_periods=window).std().reset_index(level=0,drop=True))
    evaluation_df[f"active_power_rolling_min_{window}"] = (historical_power.groupby(evaluation_df["segment_id"]).rolling(window=window,min_periods=window).min().reset_index(level=0,drop=True))
    evaluation_df[f"active_power_rolling_max_{window}"] = (historical_power.groupby(evaluation_df["segment_id"]).rolling(window=window,min_periods=window).max().reset_index(level=0,drop=True))
    

In [ ]:
rolling_columns = [
    column
    for column in evaluation_df.columns
    if "rolling_" in column
]

print("Rolling feature columns:")

for column in rolling_columns:
    print(column)

In [ ]:
test_index = 100
current_segment = evaluation_df.loc[
    test_index,
    "segment_id"
]

manual_history = evaluation_df.loc[
    (test_index - 15):(test_index - 1),
    primary_signal
]

print(
    "Current power:",
    evaluation_df.loc[test_index, primary_signal]
)

print(
    "Manual previous-15 mean:",
    manual_history.mean()
)

print(
    "Engineered rolling mean:",
    evaluation_df.loc[
        test_index,
        "active_power_rolling_mean_15"
    ]
)

In [ ]:
evaluation_df["active_power_change_1"] = (evaluation_df[primary_signal]-evaluation_df["active_power_lag_1"])
epsilon = 1e-6

evaluation_df["active_power_change_rate"] = (
    evaluation_df["active_power_change_1"]
    / (
        evaluation_df["active_power_lag_1"].abs()
        + epsilon
    )
)

In [ ]:
evaluation_df["deviation_from_15min_mean"] = (evaluation_df[primary_signal] - evaluation_df["active_power_rolling_mean_15"])
evaluation_df["deviation_from_60min_mean"] = (evaluation_df[primary_signal] - evaluation_df["active_power_rolling_mean_60"])

evaluation_df["rolling_zscore_15"] = (evaluation_df["deviation_from_15min_mean"]/(evaluation_df["active_power_rolling_std_15"]+ epsilon))
evaluation_df["rolling_zscore_60"] = (evaluation_df["deviation_from_60min_mean"]/(evaluation_df["active_power_rolling_std_60"]+ epsilon))

change_deviation_columns = ["active_power_change_1","active_power_change_rate","deviation_from_15min_mean","deviation_from_60min_mean","rolling_zscore_15","rolling_zscore_60"]
evaluation_df[["datetime","Global_active_power"]+ change_deviation_columns].head(70)

In [ ]:
for column in change_deviation_columns:

    inf_count = np.isinf(evaluation_df[column]).sum()

    print(column,"infinite values:",inf_count)

evaluation_df[change_deviation_columns].describe()

In [ ]:
evaluation_df[
    [
        "datetime",
        "segment_id",
        "Global_active_power",
        "active_power_lag_1",
        "active_power_change_1",
        "active_power_change_rate"
    ]
].sort_values(
    "active_power_change_rate",
    ascending=False
).head(10)

In [ ]:
unique_power_values = np.sort(evaluation_df[primary_signal].dropna().unique())

power_increments = np.diff(unique_power_values)

positive_increments = power_increments[power_increments > 1e-10]

print("Minimum positive power increment:",positive_increments.min())

print("Median positive power increment:",np.median(positive_increments))

print("1st percentile positive increment:",np.percentile(positive_increments,1))

In [ ]:
power_resolution = np.percentile(positive_increments,1)

std_floor = power_resolution

print("Estimated power resolution:",power_resolution)

print("Standard deviation floor:",std_floor)

In [ ]:
print("15-minute std below floor:",(evaluation_df["active_power_rolling_std_15"]< std_floor).sum())

print("60-minute std below floor:",(evaluation_df["active_power_rolling_std_60"]< std_floor).sum())

In [ ]:
stable_std_15 = (evaluation_df["active_power_rolling_std_15"].clip(lower=std_floor))
stable_std_60 = (evaluation_df["active_power_rolling_std_60"].clip(lower=std_floor))

In [ ]:
evaluation_df["rolling_zscore_15"] = (evaluation_df["deviation_from_15min_mean"]/stable_std_15)
evaluation_df["rolling_zscore_60"] = (evaluation_df["deviation_from_60min_mean"]/stable_std_60)

In [ ]:
evaluation_df[["rolling_zscore_15","rolling_zscore_60"]].describe()

In [ ]:
def signed_log_transform(series):
    return(np.sign(series)*np.log1p(np.abs(series)))

In [ ]:
evaluation_df["active_power_change_rate_log"] = (signed_log_transform(evaluation_df["active_power_change_rate"]))
evaluation_df["rolling_zscore_15_log"] = (signed_log_transform(evaluation_df["rolling_zscore_15"]))
evaluation_df["rolling_zscore_60_log"] = (signed_log_transform(evaluation_df["rolling_zscore_60"]))

In [ ]:
comparison_columns = ["active_power_change_rate","active_power_change_rate_log","rolling_zscore_15","rolling_zscore_15_log","rolling_zscore_60","rolling_zscore_60_log"]

evaluation_df[comparison_columns].describe()

In [ ]:
evaluation_df.isnull().sum()

In [ ]:
evaluation_df.isnull().sum().sort_values(ascending=False).head(20)

In [ ]:
evaluation_df = evaluation_df.dropna().copy()

In [ ]:
missing_features = [col for col in feature_columns if col not in evaluation_df.columns]
print("Missing Features:",missing_features)

In [ ]:
X_eval = evaluation_df[feature_columns]
X_eval.head()

In [ ]:
X_eval_scaled = sc.transform(X_eval)
print(X_eval_scaled.shape)

In [ ]:
predictions = iso_model.predict(X_eval_scaled)
np.unique(predictions,return_counts=True)

In [ ]:
predictions = np.where(predictions == -1,1,0)
np.unique(predictions,return_counts=True)

## Performance Discussion

The effectiveness of an unsupervised anomaly detection model cannot be assessed solely through accuracy.

Instead, evaluation considers:

- Distribution of anomaly predictions
- Consistency of anomaly scores
- Visual validation
- Practical interpretability
- Ability to detect rare consumption patterns

In [ ]:
accuracy = accuracy_score(
    evaluation_df["true_anomaly"],
    predictions
)

precision = precision_score(
    evaluation_df["true_anomaly"],
    predictions,
    zero_division=0
)

recall = recall_score(
    evaluation_df["true_anomaly"],
    predictions,
    zero_division=0
)

f1 = f1_score(
    evaluation_df["true_anomaly"],
    predictions,
    zero_division=0
)

In [ ]:
print("="*60)

print("MODEL EVALUATION")

print("="*60)

print(f"Accuracy : {accuracy:.4f}")

print(f"Precision : {precision:.4f}")

print(f"Recall : {recall:.4f}")

print(f"F1 Score : {f1:.4f}")

In [ ]:
print(
    classification_report(
        evaluation_df["true_anomaly"],
        predictions,
        zero_division=0
    )
)

In [ ]:
cm = confusion_matrix(
    evaluation_df["true_anomaly"],
    predictions
)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=[
        "Normal",
        "Anomaly"
    ]
)

plt.figure(figsize=(6,6))

disp.plot(cmap="Blues")

plt.title("Confusion Matrix")

plt.show()

In [ ]:
tn, fp, fn, tp = cm.ravel()

print(f"True Positives : {tp}")

print(f"True Negatives : {tn}")

print(f"False Positives : {fp}")

print(f"False Negatives : {fn}")

In [ ]:
false_positive_df = evaluation_df[
    (evaluation_df["true_anomaly"] == 0) &
    (predictions == 1)
]

print(false_positive_df.shape)

false_positive_df.head()

In [ ]:
false_negative_df = evaluation_df[
    (evaluation_df["true_anomaly"] == 1) &
    (predictions == 0)
]

print(false_negative_df.shape)

false_negative_df.head()

In [ ]:
evaluation_df["predicted_anomaly"] = predictions

## Hyperparameter Tuning

In [ ]:
from sklearn.ensemble import IsolationForest

In [ ]:
parameter_grid = [

    {
        "name": "Model_A",
        "n_estimators":100,
        "contamination":0.01,
        "max_samples":"auto"
    },

    {
        "name":"Model_B",
        "n_estimators":200,
        "contamination":0.01,
        "max_samples":"auto"
    },

    {
        "name":"Model_C",
        "n_estimators":300,
        "contamination":0.01,
        "max_samples":"auto"
    },

    {
        "name":"Model_D",
        "n_estimators":200,
        "contamination":0.005,
        "max_samples":"auto"
    },

    {
        "name":"Model_E",
        "n_estimators":200,
        "contamination":0.02,
        "max_samples":"auto"
    }

]
results = []

In [ ]:
model_df = evaluation_df.copy()

In [ ]:
X = model_df[feature_columns]


In [ ]:
X_scaled = sc.transform(X)

In [ ]:
from sklearn.model_selection import ParameterGrid
from sklearn.ensemble import IsolationForest
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import pandas as pd

param_grid = {
    "n_estimators": [100, 200],
    "max_samples": ["auto"],
    "contamination": [0.01, 0.02],
    "max_features": [1.0],
    "random_state": [42]
}

results = []

X = evaluation_df[feature_columns]
X_scaled = sc.transform(X)

# Ground truth: 1 = anomaly, 0 = normal
y_true = evaluation_df["true_anomaly"]

In [ ]:
for params in ParameterGrid(param_grid):

    model = IsolationForest(**params)
    model.fit(X_scaled)

    pred = model.predict(X_scaled)

    # Convert Isolation Forest output
    pred = [1 if p == -1 else 0 for p in pred]

    accuracy = accuracy_score(y_true, pred)
    precision = precision_score(y_true, pred, zero_division=0)
    recall = recall_score(y_true, pred, zero_division=0)
    f1 = f1_score(y_true, pred, zero_division=0)

    results.append({
        "n_estimators": params["n_estimators"],
        "max_samples": params["max_samples"],
        "contamination": params["contamination"],
        "max_features": params["max_features"],
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1_score": f1
    })

In [ ]:
results_df = pd.DataFrame(results)
results_df

In [ ]:
results_df = results_df.sort_values(
    by="f1_score",
    ascending=False
)

results_df

In [ ]:
best_params = results_df.iloc[0]


In [ ]:
best_iso_model = IsolationForest(
    n_estimators=int(best_params["n_estimators"]),
    max_samples=best_params["max_samples"],
    contamination=float(best_params["contamination"]),
    max_features=float(best_params["max_features"]),
    random_state=42
)

best_iso_model.fit(X_scaled)

best_predictions = best_iso_model.predict(X_scaled)

best_predictions = [1 if p == -1 else 0 for p in best_predictions]

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(
    y_true,
    best_predictions,
    zero_division=0
))

In [ ]:
evaluation_df["anomaly_score"] = (best_iso_model.decision_function(X_scaled))
evaluation_df.head()

In [ ]:
evaluation_df["best_predicted_anomaly"] = best_predictions

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

y_true = evaluation_df["true_anomaly"]
y_pred = evaluation_df["predicted_anomaly"]

print("="*60)
print("FINAL MODEL PERFORMANCE")
print("="*60)

print(f"Accuracy : {accuracy_score(y_true,y_pred):.4f}")
print(f"Precision: {precision_score(y_true,y_pred,zero_division=0):.4f}")
print(f"Recall   : {recall_score(y_true,y_pred,zero_division=0):.4f}")
print(f"F1 Score : {f1_score(y_true,y_pred,zero_division=0):.4f}")

In [ ]:
top_anomalies = (
    evaluation_df
    .sort_values("anomaly_score")
    .head(20)
)

top_anomalies

## Comparison of model

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Ground truth
y_true = evaluation_df["true_anomaly"]

# Original model predictions
original_pred = iso_model.predict(X_scaled)

# Convert Isolation Forest output
original_pred = [1 if x == -1 else 0 for x in original_pred]

original_accuracy = accuracy_score(y_true, original_pred)
original_precision = precision_score(y_true, original_pred, zero_division=0)
original_recall = recall_score(y_true, original_pred, zero_division=0)
original_f1 = f1_score(y_true, original_pred, zero_division=0)

In [ ]:
# Tuned model predictions
tuned_pred = best_iso_model.predict(X_scaled)

tuned_pred = [1 if x == -1 else 0 for x in tuned_pred]

tuned_accuracy = accuracy_score(y_true, tuned_pred)
tuned_precision = precision_score(y_true, tuned_pred, zero_division=0)
tuned_recall = recall_score(y_true, tuned_pred, zero_division=0)
tuned_f1 = f1_score(y_true, tuned_pred, zero_division=0)

In [ ]:
comparison_df = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1-Score"],
    "Initial Model": [
        original_accuracy,
        original_precision,
        original_recall,
        original_f1
    ],
    "Hyperparameter Tuned Model": [
        tuned_accuracy,
        tuned_precision,
        tuned_recall,
        tuned_f1
    ]
})

comparison_df

In [ ]:
comparison_df["Improvement"] = (
    comparison_df["Hyperparameter Tuned Model"] -
    comparison_df["Initial Model"]
)

comparison_df

In [ ]:
comparison_df.style.format({
    "Initial Model": "{:.4f}",
    "Hyperparameter Tuned Model": "{:.4f}",
    "Improvement": "{:+.4f}"
})

## Model Outputs

The evaluation process generates:

- Final anomaly predictions
- Anomaly scores
- Performance visualizations
- Evaluation summaries

These outputs are integrated into the EcoWatt AI dashboard for interactive exploration.

In [ ]:
import joblib

# Save the trained model
joblib.dump(best_iso_model, "best_isolation_forest_model.pkl")

# Save the scaler
joblib.dump(sc, "standard_scaler.pkl")

print("✅ Best model and scaler saved successfully!")

In [ ]:
# Predict anomalies
final_predictions = best_iso_model.predict(X_scaled)

# Convert predictions
evaluation_df["final_predicted_anomaly"] = [
    1 if x == -1 else 0 for x in final_predictions
]

# Calculate anomaly score
evaluation_df["final_anomaly_score"] = best_iso_model.decision_function(X_scaled)

evaluation_df.head()

In [ ]:
evaluation_df.to_csv(
    "data/processed/final_energy_anomaly_results.csv",
    index=False
)

print("✅ Final results saved.")

In [ ]:
print(evaluation_df["final_predicted_anomaly"].value_counts())

In [ ]:
import matplotlib.pyplot as plt

counts = evaluation_df["final_predicted_anomaly"].value_counts()

plt.figure(figsize=(6,5))

plt.bar(
    ["Normal", "Anomaly"],
    [counts[0], counts[1]]
)

plt.title("Detected Normal vs Anomalous Records")
plt.xlabel("Class")
plt.ylabel("Number of Records")

plt.show()

In [ ]:
plt.figure(figsize=(10,5))

plt.hist(
    evaluation_df["final_anomaly_score"],
    bins=50
)

plt.title("Distribution of Anomaly Scores")
plt.xlabel("Anomaly Score")
plt.ylabel("Frequency")

plt.show()

In [ ]:
top20 = evaluation_df.sort_values(
    by="anomaly_score"
).head(20)

top20

In [ ]:
top20.to_csv(
    "C:/EcoWatt-AI/data/processed/top_20_energy_anomalies.csv",
    index=False
)

In [ ]:
summary = evaluation_df.describe()

summary

In [ ]:
summary.to_csv("data/processed/dataset_summary.csv")

# Why Isolation Forest?

Isolation Forest was selected because it offers several advantages for anomaly detection:

- Does not require labelled training data.
- Efficient for large datasets.
- Performs well with high-dimensional features.
- Naturally isolates rare observations.
- Scales effectively for real-world applications.
- Suitable for energy monitoring systems where abnormal events are infrequent.

# Limitations

Although Isolation Forest performs effectively for anomaly detection, several limitations should be considered:

- The model does not identify the exact cause of an anomaly.
- Performance depends on the quality of engineered features.
- The contamination parameter influences the number of detected anomalies.
- Results should be interpreted alongside domain knowledge.

# Key Takeaways

- The Isolation Forest model successfully identified abnormal household energy consumption patterns.
- Anomaly scores provided additional information beyond binary predictions.
- Visual analysis supported the validity of detected anomalies.
- The evaluation confirmed that the generated features effectively represent household electricity consumption behaviour.
- The trained model is suitable for deployment in the EcoWatt AI dashboard.

# Conclusion

The Isolation Forest model demonstrated its capability to detect unusual household electricity consumption without requiring labelled training data.

Through anomaly prediction, anomaly score analysis, and visualization, the model successfully identified observations that deviated from normal consumption behaviour.

The evaluation confirms that the combination of preprocessing, feature engineering, and Isolation Forest provides a robust framework for anomaly detection.

The validated model is now ready for explainability analysis using SHAP and deployment within the EcoWatt AI interactive dashboard.